# Lesson 22: Decision Theory and Causal Inference

## Opening Story: Personalized Treatment Rules

A doctor has a patient with high cholesterol. Should they prescribe statins? The decision depends on the patient's risk factors, potential side effects, and the expected benefit. This is a decision-theoretic problem: what treatment should we choose to maximize expected utility?

Decision theory provides the framework for optimal treatment assignment, connecting causal inference to actionable recommendations.

---

## Learning Objectives

By the end of this lesson, you should be able to:

1. Define optimal treatment rules
2. Explain the value of information
3. Implement policy learning
4. Conduct cost-benefit analysis
5. Connect causal inference to decision-making

---

## 22.1 Optimal Treatment Rules

### Definition

A treatment rule $\delta(X)$ assigns treatment based on observed covariates:

$$\delta(X) = \arg\max_{a \in \{0,1\}} E[Y(a) | X]$$

### Value of a Rule

The value of a treatment rule is the expected outcome if we follow it:

$$V(\delta) = E[Y^{\delta(X)}]$$

---

## 22.2 Policy Learning

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
n = 1000
p = 5

# Generate data
X = np.random.normal(0, 1, (n, p))
T = np.random.binomial(1, 0.5, n)
Y = 2 * T + X @ np.ones(p) + np.random.normal(0, 1, n)

# Heterogeneous effects
true_effect = 2 + X[:, 0] - X[:, 1]

# Optimal rule: treat if effect > 0
optimal_rule = (true_effect > 0).astype(int)

# Value of optimal rule
value_optimal = Y[optimal_rule == 1].mean() * optimal_rule.mean() + \
                Y[optimal_rule == 0].mean() * (1 - optimal_rule).mean()

# Value of treat-all
value_all = Y.mean()

# Value of treat-none
value_none = Y[T == 0].mean()

print(f"Value of treat-all: {value_all:.3f}")
print(f"Value of treat-none: {value_none:.3f}")
print(f"Value of optimal rule: {value_optimal:.3f}")

# Learn a policy using treatment effects
from econml.metalearners import XLearner
from sklearn.ensemble import RandomForestRegressor

x_learner = XLearner(models=RandomForestRegressor(n_estimators=100, random_state=42))
x_learner.fit(Y, T, X=X)
te_pred = x_learner.effect(X)

# Learned rule: treat if predicted effect > 0
learned_rule = (te_pred > 0).astype(int)
value_learned = Y[learned_rule == 1].mean() * learned_rule.mean() + \
                Y[learned_rule == 0].mean() * (1 - learned_rule).mean()

print(f"Value of learned rule: {value_learned:.3f}")

---

## 22.3 Cost-Benefit Analysis

In [ ]:
# Include costs of treatment
cost_treatment = 100
benefit_treatment = 500  # Monetary value of health improvement

# Net benefit
def net_benefit(rule, Y, cost):
    treated = rule.sum()
    benefit = Y[rule == 1].sum() - cost * treated
    return benefit

# Compare policies
print(f"\nNet benefit (optimal): {net_benefit(optimal_rule, Y, cost_treatment):.2f}")
print(f"Net benefit (learned): {net_benefit(learned_rule, Y, cost_treatment):.2f}")
print(f"Net benefit (treat all): {net_benefit(np.ones(n), Y, cost_treatment):.2f}")

---

## 22.4 Common Mistakes

1. **Ignoring heterogeneity**: Treat-all is suboptimal when effects vary
2. **Overfitting policies**: Use cross-fitting for valid estimation
3. **Ignoring costs**: Benefits must outweigh costs
4. **Short-term vs long-term**: Consider time horizons

---

## 22.5 Knowledge Check

### Multiple Choice

1. **An optimal treatment rule:**
   A) Treats everyone
   B) Treats no one
   C) Assigns treatment based on covariates
   D) Randomizes treatment

2. **Policy learning:**
   A) Estimates average effects
   B) Learns treatment assignment rules
   C) Tests causal hypotheses
   D) All of the above

3. **The value of a rule is:**
   A) The average treatment effect
   B) The expected outcome under the rule
   C) The probability of treatment
   D) The variance of outcomes

4. **Cost-benefit analysis:**
   A) Ignores costs
   B) Considers both costs and benefits
   C) Only considers benefits
   D) Only considers costs

5. **Cross-fitting in policy learning:**
   A) Increases bias
   B) Reduces overfitting
   C) Has no effect
   D) Speeds computation

### Short Answer

6. **Explain how causal inference informs treatment decisions.**

7. **What is the difference between treat-all and an optimal rule?**

8. **How do you evaluate a learned treatment rule?**

9. **Why is cross-fitting important for policy learning?**

10. **Give an example where personalized treatment rules would be valuable.**

---

## 22.6 Summary

1. **Optimal treatment rules** assign treatment based on covariates
2. **Policy learning** estimates these rules from data
3. **Value assessment** compares rules to benchmarks
4. **Cost-benefit analysis** incorporates real-world constraints
5. **Cross-fitting** ensures valid policy evaluation

---

## 22.7 Further Reading

- Robins, J.M. (2000). "Optimal Structural Nested Models for Optimal Sequential Decisions."
- Kitagawa, T. & Tetenov, A. (2018). "Who Should Be Treated? Optimal Treatment Assignment Rules." *Econometrics*.